In [1]:
import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import NETWORK_TYPE, extract_unique_npcis, RF_PARAM_5G

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt('data/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ['toa_pps', 'toa_cir', 'toa_cov', 'campaign_id']
df['measurements_matrix'] = df['measurements_matrix'].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

selected_campaigns = list(range(1, 10))
# Data filtering
df = filter_dataframe(
    df=df,
    operators=[10],
    include_columns=['pci', 'beam_index', 'nr_arfcn', 'operator_id', 'rsrq'],
    campaigns=selected_campaigns,
)

Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [12]:
#df_tp, df_rp = dataset_tp_rp_split(df, 0.3, 42)

unique_npcis = extract_unique_npcis(df['measurements_matrix'])
rf_param = RF_PARAM_5G.RSRQ
n_clusters = 5
random_seed = 42

In [15]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


def evaluate_clustering(df, kmeans_model):
    coords = df[["lat", "lng"]].values
    cluster_labels = kmeans_model.labels_

    # Calculate silhouette score (ranges from -1 to 1, higher is better)
    sil_score = silhouette_score(coords, cluster_labels)
    print(f"Silhouette Score: {sil_score:.4f}")

    # Calculate inertia (sum of squared distances to closest centroid)
    inertia = kmeans_model.inertia_
    print(f"Inertia: {inertia:.4f}")

    return sil_score, inertia


#df_features, _ = create_point_matrix(df, unique_npcis, rf_param)
coords = df[["lat", "lng"]].values

kmeans = KMeans(n_clusters=n_clusters, random_state=random_seed)

df["cluster"] = kmeans.fit_predict(coords)

res = evaluate_clustering(df, kmeans)

Silhouette Score: 0.6148
Inertia: 0.0007
